# Adding and Subtracting Fractions with Unlike Denominators

In the previous notebook, adding fractions was simple because the denominators matched — same chunk size, just count the chunks.

When the denominators differ, you have a unit mismatch. You need to convert both fractions to the same unit before you can combine them. This notebook shows how to find that common unit efficiently using the **Least Common Multiple (LCM)**.

**Prerequisites:** Notebooks 01–04.

---

## Intuition: Unit Mismatch

You cannot add `1/3` and `1/4` directly for the same reason you cannot add 1 metre and 1 foot without converting — the units are different.

```python
# Can't do this without conversion:
metres + feet  # TypeError in a typed system — incompatible units

# You'd convert both to centimetres first:
metres_in_cm + feet_in_cm  # now they're the same unit
```

With fractions, the "unit" is the denominator. `1/3` means thirds; `1/4` means quarters. To add them you need a denominator that both thirds and quarters fit into evenly — a common multiple.

Any common multiple works mathematically. The **Least** Common Multiple keeps the numbers as small as possible, which means less simplification at the end.

---

## The LCM: Smallest Shared Multiple

The LCM of two numbers is the smallest number that both divide into evenly.

```
Multiples of 3:  3, 6, 9, 12, 15, 18 ...
Multiples of 4:  4, 8, 12, 16, 20 ...
                       ^
                 First match: 12  =>  LCM(3, 4) = 12
```

Why not just multiply the denominators? That would give 3 × 4 = 12 here, which happens to be the same. But for `LCM(4, 6)`, the product gives 24 while the LCM is 12. Using 12 keeps the numerators smaller and avoids extra simplification.

The efficient algorithm uses the GCD (Greatest Common Divisor):

$$\text{LCM}(a, b) = \frac{|a \times b|}{\text{GCD}(a, b)}$$

In [ ]:
from math import gcd

def lcm(a, b):
    return abs(a * b) // gcd(a, b)

# Show why LCM beats the product for some pairs
pairs = [(3, 4), (4, 6), (5, 10), (6, 8), (7, 3)]

print(f"{'a':>4}  {'b':>4}  {'a*b':>6}  {'LCM':>6}  {'Saving?'}")
print("-" * 38)
for a, b in pairs:
    product = a * b
    l = lcm(a, b)
    saving = f"product is {product // l}x larger" if product != l else "same"
    print(f"{a:>4}  {b:>4}  {product:>6}  {l:>6}  {saving}")

---

## The Algorithm: Step by Step

To add or subtract `a/b` and `c/d`:

1. Find `common = LCM(b, d)`
2. Scale the first fraction: multiply top and bottom by `common // b`
3. Scale the second fraction: multiply top and bottom by `common // d`
4. Add or subtract the numerators
5. Simplify the result

Scaling a fraction by `n/n` is multiplying by 1 — the value doesn't change, only the representation.

In [ ]:
from math import gcd

def lcm(a, b):
    return abs(a * b) // gcd(a, b)

def add_fractions_trace(a_num, a_den, b_num, b_den, op='+'):
    """Add or subtract two fractions, printing each step."""
    op_sign = 1 if op == '+' else -1

    print(f"  {a_num}/{a_den}  {op}  {b_num}/{b_den}")

    common = lcm(a_den, b_den)
    print(f"  LCM({a_den}, {b_den}) = {common}")

    scale_a = common // a_den
    scale_b = common // b_den
    new_a_num = a_num * scale_a
    new_b_num = b_num * scale_b

    print(f"  Scale first:  {a_num}/{a_den} x {scale_a}/{scale_a} = {new_a_num}/{common}")
    print(f"  Scale second: {b_num}/{b_den} x {scale_b}/{scale_b} = {new_b_num}/{common}")

    result_num = new_a_num + op_sign * new_b_num
    print(f"  Combine:      {new_a_num}/{common} {op} {new_b_num}/{common} = {result_num}/{common}")

    divisor = gcd(abs(result_num), common)
    simplified = f"{result_num // divisor}/{common // divisor}"
    print(f"  Simplified:   {simplified}")
    print()

add_fractions_trace(2, 3, 1, 4, '+')
add_fractions_trace(-5, 6, 1, 8, '-')
add_fractions_trace(3, 4, -1, 6, '+')

---

## Visualisation: Finding the Common Denominator

The conversion step — scaling each fraction to the common denominator — is easiest to see with fraction bars. Each bar is divided into equal segments; the scaling just subdivides them until both have the same segment size.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from math import gcd

def lcm(a, b):
    return abs(a * b) // gcd(a, b)

def plot_common_denominator(a_num, a_den, b_num, b_den):
    common = lcm(a_den, b_den)
    scale_a = common // a_den
    scale_b = common // b_den
    new_a = a_num * scale_a
    new_b = b_num * scale_b

    fig, axes = plt.subplots(2, 2, figsize=(12, 4))
    bar_h = 0.5

    def draw_bar(ax, numerator, denominator, colour, title):
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_yticks([])
        ax.set_title(title, fontsize=10)
        # Full bar outline
        ax.add_patch(patches.Rectangle((0, 0.25), 1, bar_h,
                                       linewidth=1.5, edgecolor='black', facecolor='whitesmoke'))
        # Filled portion
        fill_width = abs(numerator) / denominator
        ax.add_patch(patches.Rectangle((0, 0.25), fill_width, bar_h,
                                       linewidth=0, facecolor=colour, alpha=0.7))
        # Segment dividers
        for i in range(1, denominator):
            x = i / denominator
            ax.axvline(x, color='black', linewidth=0.5, ymin=0.25, ymax=0.75)
        ax.set_xticks([i / denominator for i in range(denominator + 1)])
        ax.set_xticklabels(
            [f"{i}/{denominator}" if i > 0 else "0" for i in range(denominator + 1)],
            fontsize=7
        )

    draw_bar(axes[0, 0], a_num, a_den, 'steelblue', f'Original: {a_num}/{a_den}')
    draw_bar(axes[0, 1], b_num, b_den, 'darkorange', f'Original: {b_num}/{b_den}')
    draw_bar(axes[1, 0], new_a, common, 'steelblue',
             f'Converted: {new_a}/{common}  (x{scale_a})')
    draw_bar(axes[1, 1], new_b, common, 'darkorange',
             f'Converted: {new_b}/{common}  (x{scale_b})')

    result_num = new_a + new_b
    d = gcd(abs(result_num), common)
    simplified = f"{result_num // d}/{common // d}"
    fig.suptitle(
        f'{a_num}/{a_den} + {b_num}/{b_den}'
        f'  =>  {new_a}/{common} + {new_b}/{common}'
        f'  =  {result_num}/{common}  =  {simplified}',
        fontsize=11
    )
    plt.tight_layout()
    plt.show()

plot_common_denominator(2, 3, 1, 4)
plot_common_denominator(1, 4, 1, 6)

---

## Using Python's `Fraction` Directly

Python's `fractions.Fraction` handles all of this automatically and exactly — no floating-point rounding, no manual LCM. It's worth knowing how to use it, both as a calculator and as a sanity check.

In [ ]:
from fractions import Fraction

problems = [
    (Fraction(2, 3),  '+', Fraction(1, 4)),
    (Fraction(-5, 6), '-', Fraction(1, 8)),
    (Fraction(3, 4),  '+', Fraction(-1, 6)),
    (Fraction(7, 10), '-', Fraction(-3, 4)),
]

for a, op, b in problems:
    result = a + b if op == '+' else a - b
    print(f"  {str(a):>6}  {op}  {str(b):<6}  =  {result}")

---

## Exercises

1. Calculate $\frac{3}{8} + \frac{5}{12}$ by hand using the LCM method, then verify with `Fraction`.
2. What is `LCM(a, a)` for any `a`? Why? Does the algorithm confirm this?
3. Add three fractions: $\frac{1}{2} + \frac{1}{3} + \frac{1}{6}$. Extend `add_fractions_trace` to handle three inputs, or chain two calls.
4. The `add_fractions_trace` function assumes the denominator is always positive. What happens if you pass a negative denominator like `add_fractions_trace(1, -3, 1, 4, '+')`? Fix it.

In [ ]:
# Your experiments here


---

## Formal Notation

For fractions with different denominators:

$$\frac{a}{b} + \frac{c}{d} = \frac{a \cdot \frac{\text{lcm}(b,d)}{b} + c \cdot \frac{\text{lcm}(b,d)}{d}}{\text{lcm}(b,d)}$$

Which simplifies to the more commonly written form using the product as the common denominator:

$$\frac{a}{b} + \frac{c}{d} = \frac{ad + bc}{bd}$$

Both are correct. The LCM form produces a smaller denominator; the product form is easier to memorise. `fractions.Fraction` uses the GCD/LCM approach internally.

---

## Real-World Connection

- **Time arithmetic**: adding durations in different units (`3/4` of an hour + `1/3` of an hour) is fraction addition with unlike denominators. The LCM of 4 and 3 is 12 — so you work in twelfths of an hour (5-minute increments).
- **Mixing ratios**: combining two solutions at different concentrations requires adding fractions. A recipe calling for `2/3` cup of one ingredient and `3/4` cup of another needs a common unit to calculate the total.
- **Rational arithmetic in compilers**: exact rational number libraries (used in computer algebra systems and some language runtimes) implement this exact algorithm. The `fractions.Fraction` class in Python's standard library is a production-quality version of what we built above.
- **Probability**: adding probabilities of mutually exclusive events expressed as fractions with different denominators — `P(A) = 1/3`, `P(B) = 1/4`, `P(A or B) = ?` — is this exact operation.

---

## Summary

- Unlike denominators are a unit mismatch — convert both fractions to a common unit before combining
- The Least Common Multiple gives the smallest common denominator, keeping numbers manageable
- `LCM(a, b) = |a * b| / GCD(a, b)` — efficient to compute
- Algorithm: find LCM, scale each fraction's numerator by `LCM / denominator`, add numerators, simplify
- Python's `fractions.Fraction` does all of this exactly — use it to verify hand calculations
- The sign rules from notebook 04 apply unchanged; the LCM step just establishes the common denominator first